# LinkedIn User Prediction Model
## Building a Machine Learning Classification Model

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

## Step 1: Read in the data and check dimensions

In [ ]:
# Read the data into dataframe 's'
s = pd.read_csv('social_media_usage.csv')

# Check dimensions
print(f"Dataset shape: {s.shape}")
print(f"Rows: {s.shape[0]}, Columns: {s.shape[1]}")

## Step 2: Define and test clean_sm function

In [ ]:
# Define clean_sm function
def clean_sm(x):
    x = np.where(x == 1, 1, 0)
    return x

In [ ]:
# Test with toy dataframe
toy_df = pd.DataFrame({
    'col1': [1, 2, 1],
    'col2': [1, 1, 8]
})

print("Before:")
print(toy_df)

print("\nAfter clean_sm:")
toy_df['col1_clean'] = clean_sm(toy_df['col1'])
toy_df['col2_clean'] = clean_sm(toy_df['col2'])
print(toy_df)

## Step 3: Create ss dataframe and perform exploratory analysis

In [ ]:
# Create ss dataframe
ss = pd.DataFrame({
    'sm_li': clean_sm(s['web1h']),
    'income': np.where(s['income'] > 9, np.nan, s['income']),
    'education': np.where(s['educ2'] > 8, np.nan, s['educ2']),
    'parent': np.where(s['par'] == 1, 1, np.where(s['par'] == 2, 0, np.nan)),
    'married': np.where(s['marital'] == 1, 1, np.where(s['marital'] == 2, 0, np.nan)),
    'female': np.where(s['gender'] == 2, 1, np.where(s['gender'] == 1, 0, np.nan)),
    'age': np.where(s['age'] > 98, np.nan, s['age'])
})

# Drop missing values
ss = ss.dropna()

print(f"Clean dataset shape: {ss.shape}")
print(f"\nFirst few rows:")
print(ss.head())

In [ ]:
# Summary statistics
print(ss.describe())

In [ ]:
# Target variable distribution
print("LinkedIn usage distribution:")
print(ss['sm_li'].value_counts())
print(f"\nPercentage using LinkedIn: {ss['sm_li'].mean() * 100:.2f}%")

In [ ]:
# Income vs LinkedIn usage
plt.figure(figsize=(10, 6))
ss.groupby('income')['sm_li'].mean().plot(kind='bar')
plt.xlabel('Income Level')
plt.ylabel('Proportion Using LinkedIn')
plt.title('LinkedIn Usage by Income Level')
plt.show()

In [ ]:
# Education vs LinkedIn usage
plt.figure(figsize=(10, 6))
ss.groupby('education')['sm_li'].mean().plot(kind='bar', color='green')
plt.xlabel('Education Level')
plt.ylabel('Proportion Using LinkedIn')
plt.title('LinkedIn Usage by Education Level')
plt.show()

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(ss.corr(), annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

## Step 4: Create target vector (y) and feature set (X)

In [ ]:
# Features
X = ss[['income', 'education', 'parent', 'married', 'female', 'age']]

# Target
y = ss['sm_li']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## Step 5: Split data into training and test sets

**What each object contains:**
- **X_train**: Training features (80% of data) - used to train the model
- **X_test**: Test features (20% of data) - used to evaluate the model
- **y_train**: Training target values - actual LinkedIn usage for training data
- **y_test**: Test target values - actual LinkedIn usage for test data

**How they're used in machine learning:**
- The model learns patterns from X_train and y_train
- We evaluate the model on X_test and y_test (data it hasn't seen)
- This tests if the model can generalize to new data

In [ ]:
# Split: 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Step 6: Instantiate and fit logistic regression model

In [ ]:
# Instantiate model
lr = LogisticRegression(class_weight='balanced', random_state=42)

# Fit model
lr.fit(X_train, y_train)

print("Model trained successfully!")

## Step 7: Evaluate model and generate confusion matrix

In [ ]:
# Make predictions
y_pred = lr.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

print("\nInterpretation:")
print(f"True Negatives (TN): {cm[0, 0]} - Correctly predicted NOT LinkedIn users")
print(f"False Positives (FP): {cm[0, 1]} - Incorrectly predicted as LinkedIn users")
print(f"False Negatives (FN): {cm[1, 0]} - Incorrectly predicted as NOT LinkedIn users")
print(f"True Positives (TP): {cm[1, 1]} - Correctly predicted LinkedIn users")

## Step 8: Create confusion matrix as dataframe

In [ ]:
# Confusion matrix as dataframe
cm_df = pd.DataFrame(cm,
                     index=['Actual: No LinkedIn', 'Actual: Uses LinkedIn'],
                     columns=['Predicted: No LinkedIn', 'Predicted: Uses LinkedIn'])

print(cm_df)

In [ ]:
# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True, square=True)
plt.title('Confusion Matrix')
plt.show()

## Step 9: Calculate precision, recall, and F1 score by hand

In [ ]:
# Extract values
tn, fp, fn, tp = cm.ravel()

print(f"TN = {tn}, FP = {fp}, FN = {fn}, TP = {tp}")

In [ ]:
# Calculate precision by hand
precision = tp / (tp + fp)
print(f"Precision = TP / (TP + FP) = {tp} / ({tp} + {fp}) = {precision:.4f}")
print(f"\nInterpretation: Of all predicted LinkedIn users, {precision * 100:.2f}% actually use it.")
print("When to use: When false positives are costly (e.g., spam detection).")

In [ ]:
# Calculate recall by hand
recall = tp / (tp + fn)
print(f"Recall = TP / (TP + FN) = {tp} / ({tp} + {fn}) = {recall:.4f}")
print(f"\nInterpretation: Of all actual LinkedIn users, we correctly identified {recall * 100:.2f}%.")
print("When to use: When false negatives are costly (e.g., disease screening).")

In [ ]:
# Calculate F1 score by hand
f1 = 2 * (precision * recall) / (precision + recall)
print(f"F1 Score = 2 × (Precision × Recall) / (Precision + Recall)")
print(f"         = 2 × ({precision:.4f} × {recall:.4f}) / ({precision:.4f} + {recall:.4f})")
print(f"         = {f1:.4f}")
print(f"\nInterpretation: Harmonic mean balancing precision and recall.")
print("When to use: When both false positives and negatives matter equally.")

In [ ]:
# Verify with sklearn
print("\nVerification with sklearn:")
print(classification_report(y_test, y_pred, target_names=['No LinkedIn', 'Uses LinkedIn']))

print(f"\nManual calculations match sklearn: ✓")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

## Step 10: Make predictions for specific individuals

In [ ]:
# Person 1: Income=8, Education=7, Non-parent, Married, Female, Age=42
person1 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [42]
})

prob1 = lr.predict_proba(person1)[0][1]
print("Person 1: High income, High education, Non-parent, Married, Female, Age 42")
print(f"Probability of using LinkedIn: {prob1:.4f} ({prob1 * 100:.2f}%)")

In [ ]:
# Person 2: Same but Age=82
person2 = pd.DataFrame({
    'income': [8],
    'education': [7],
    'parent': [0],
    'married': [1],
    'female': [1],
    'age': [82]
})

prob2 = lr.predict_proba(person2)[0][1]
print("Person 2: Same characteristics but Age 82")
print(f"Probability of using LinkedIn: {prob2:.4f} ({prob2 * 100:.2f}%)")

In [ ]:
# Compare
print("\nComparison:")
print(f"Person 1 (Age 42): {prob1:.4f} ({prob1 * 100:.2f}%)")
print(f"Person 2 (Age 82): {prob2:.4f} ({prob2 * 100:.2f}%)")
print(f"\nDifference: {prob1 - prob2:.4f} ({(prob1 - prob2) * 100:.2f} percentage points)")
print(f"\nThe 42-year-old is {prob1 / prob2:.2f}x more likely to use LinkedIn.")